<a href="https://colab.research.google.com/github/mamezquita/PROYECTO-EEP-LEARNING-NOTICIAS/blob/main/analisis_convergencia_bayesiana.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Análisis del modelo final: pérdida, gradientes y convergencia de hiperparámetros

**Proyecto Integrador · Deep Learning · 2026-2**

Notebook complementario a `finetuning_bayesiano_colombia.ipynb`. No vuelve a
correr la búsqueda bayesiana: toma los mejores hiperparámetros que esa
búsqueda encontró (guardados en `salida_bayesiana/mejores_hiperparametros.json`)
y hace dos cosas separadas con ellos:

1. **Entrena un único modelo final** con esos hiperparámetros y con la
   **misma semilla** que usó la búsqueda bayesiana (`CONFIG["seed"]`), y
   analiza su comportamiento durante el entrenamiento: pérdida de
   entrenamiento vs. validación por época, y la norma del gradiente por
   época (para detectar gradientes que se desvanecen o explotan).
2. **Por separado**, en una sección aparte (§14), reentrena esos mismos
   hiperparámetros varias veces con semillas *distintas* de la de arriba,
   para comprobar que el resultado de la búsqueda es una señal real y no
   un intento con suerte. Esta sección es la única parte del notebook que
   usa más de una semilla.

## Requisitos

1. Haber corrido `finetuning_bayesiano_colombia.ipynb` hasta el final (§19),
   de modo que exista `salida_bayesiana/mejores_hiperparametros.json`.
2. Subir ese archivo (junto con `dataset_politica_colombiana.xlsx`) cuando
   este notebook lo pida.
3. GPU activada (Entorno de ejecución → Cambiar tipo de entorno de
   ejecución → GPU).


## §1 · Instalación e importaciones

In [1]:
import sys

# TF_USE_LEGACY_KERAS solo funciona si se fija ANTES de que tensorflow se
# importe por primera vez en este proceso de kernel.
assert "tensorflow" not in sys.modules, (
    "tensorflow ya estaba importado en este kernel antes de correr esta "
    "celda. Anda a Entorno de ejecucion -> Reiniciar entorno de ejecucion, "
    "y corre esta celda primero, antes que cualquier otra."
)

# Mismas versiones fijadas que finetuning_bayesiano_colombia.ipynb, para
# que el modelo cargado desde salida_bayesiana/ sea compatible.
!pip install -q "transformers>=4.44,<5.0" "tf-keras" "keras-tuner==1.3.5" "huggingface_hub>=0.24" openpyxl scikit-learn

import os
os.environ["TF_USE_LEGACY_KERAS"] = "1"

import json
import time
import random
import re
import unicodedata
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import tensorflow as tf

print("tensorflow    :", tf.__version__)

_keras_modulo = tf.keras.__name__
_es_legado = _keras_modulo.startswith("tf_keras")
print("tf.keras modulo:", _keras_modulo, "(legado)" if _es_legado else "(Keras 3 -- NO esperado)")

assert _es_legado, (
    f"tf.keras esta resolviendo a '{_keras_modulo}', o sea Keras 3, no al modo "
    "legado -- probablemente porque el paquete tf-keras no se instalo bien. "
    "Reinicia el entorno de ejecucion y vuelve a correr esta celda primero."
)

import keras_tuner as kt

import sklearn
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    matthews_corrcoef, confusion_matrix,
)

import transformers

_transformers_mayor = int(transformers.__version__.split(".")[0])
assert _transformers_mayor < 5, (
    f"transformers {transformers.__version__} no trae clases TF. El pip "
    "install de arriba deberia haber fijado <5.0."
)

from transformers import AutoTokenizer, AutoConfig, TFAutoModelForSequenceClassification
from huggingface_hub import list_repo_files, model_info

print("keras_tuner   :", kt.__version__)
print("transformers  :", transformers.__version__)
print("scikit-learn  :", sklearn.__version__)
gpus = tf.config.list_physical_devices("GPU")
print("GPU           :", gpus if gpus else "NO DISPONIBLE (activa GPU en Entorno de ejecucion -> Cambiar tipo)")


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 176.1/176.1 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 70.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 30.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 49.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.26.0 requires huggingface-hub<2.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
diffusers 0.40.0 requires huggingface-hub<2.0,>=1.23.0, but you have huggingface-hub 0.36.2 which is incompatible.
tensorflow    : 2.20.0
tf.keras modulo: tf_keras.api._v2.keras (legado)
keras_tuner   : 1.3.5
transformers  : 4.57.6
scikit-learn  : 1.6.1
GPU           : [PhysicalDevice(name='/physical_device:GPU:0',

## §2 · Subir el corpus y los hiperparámetros encontrados

In [2]:
RUTA_XLSX = "/content/dataset_politica_colombiana.xlsx"
RUTA_HIPERPARAMETROS = "/content/mejores_hiperparametros.json"

def _asegurar_archivo(ruta, etiqueta):
    if not os.path.exists(ruta):
        try:
            from google.colab import files
            print(f"Sube '{ruta}' ({etiqueta}):")
            subido = files.upload()
            if ruta not in subido and len(subido) == 1:
                os.rename(next(iter(subido)), ruta)
        except ImportError:
            raise FileNotFoundError(
                f"No se encontró '{ruta}' y esto no parece ser Colab. Copia "
                "el archivo al directorio de trabajo manualmente."
            )
    assert os.path.exists(ruta), f"'{ruta}' sigue sin existir tras el intento de subida."
    print(f"✓ {ruta} listo ({os.path.getsize(ruta)/1024:.0f} KB)")


_asegurar_archivo(RUTA_XLSX, "corpus")
_asegurar_archivo(RUTA_HIPERPARAMETROS, "salida_bayesiana/mejores_hiperparametros.json de la búsqueda")

with open(RUTA_HIPERPARAMETROS, encoding="utf-8") as f:
    _artefacto_busqueda = json.load(f)

MEJORES_HP = _artefacto_busqueda["mejores_hiperparametros"]
print("\nHiperparámetros cargados desde la búsqueda bayesiana:\n")
for nombre, valor in MEJORES_HP.items():
    print(f"  {nombre:<28} {valor}")


Sube 'dataset_politica_colombiana.xlsx' (corpus):


Saving dataset_politica_colombiana.xlsx to dataset_politica_colombiana.xlsx
✓ dataset_politica_colombiana.xlsx listo (537 KB)
Sube 'mejores_hiperparametros.json' (salida_bayesiana/mejores_hiperparametros.json de la búsqueda):


Saving mejores_hiperparametros.json to mejores_hiperparametros.json
✓ mejores_hiperparametros.json listo (1 KB)

Hiperparámetros cargados desde la búsqueda bayesiana:

  learning_rate                4.0983871094082436e-05
  dropout                      0.07258496356733739
  weight_decay                 0.008547485565344062
  batch_size                   8
  lr_reduce_factor             0.3323028659735827
  lr_reduce_patience           1
  early_stopping_patience      3


## §3 · Configuración

In [3]:
CONFIG = {
    # ---------- DATOS ----------
    "ruta_xlsx": RUTA_XLSX,
    "col_label": "label",
    "col_titulo": "title",
    "col_descripcion": "description",
    "col_fecha": "date",
    "campos_texto": ["title", "description"],
    "etiquetas": {"0": "falsa", "1": "real"},

    # ---------- PARTICIÓN (idéntico criterio al notebook de la búsqueda) ----------
    "proporciones": [0.70, 0.15, 0.15],   # train / val / test
    "agrupar_variantes": True,

    # ---------- MODELO ----------
    "modelo_candidatos": [
        "PlanTL-GOB-ES/roberta-base-bne",
        "BSC-LT/roberta-base-bne",
        "BSC-TeMU/roberta-base-bne",
        "IsGarrido/roberta-base-bne",
    ],
    "modelo_respaldo": "bertin-project/bertin-roberta-base-spanish",
    "max_seq_length": 128,

    # ---------- ENTRENAMIENTO DEL MODELO FINAL ----------
    "epocas_techo": 15,   # mismo techo que usó la búsqueda; la parada temprana decide la duración real

    # ---------- REPRODUCIBILIDAD ----------
    # La MISMA semilla que usó la búsqueda bayesiana (Tabla 3 del artículo).
    # Todo este notebook usa esta única semilla, EXCEPTO la prueba de
    # convergencia de §14, que es la única sección que varía la semilla.
    "seed": 16,

    # ---------- PRUEBA DE CONVERGENCIA (§14) ----------
    "n_repeticiones_convergencia": 3,
}

DIR_SALIDA = "salida_bayesiana/convergencia"
os.makedirs(DIR_SALIDA, exist_ok=True)

print(json.dumps(CONFIG, indent=2, ensure_ascii=False))


{
  "ruta_xlsx": "dataset_politica_colombiana.xlsx",
  "col_label": "label",
  "col_titulo": "title",
  "col_descripcion": "description",
  "col_fecha": "date",
  "campos_texto": [
    "title",
    "description"
  ],
  "etiquetas": {
    "0": "falsa",
    "1": "real"
  },
  "proporciones": [
    0.7,
    0.15,
    0.15
  ],
  "agrupar_variantes": true,
  "modelo_candidatos": [
    "PlanTL-GOB-ES/roberta-base-bne",
    "BSC-LT/roberta-base-bne",
    "BSC-TeMU/roberta-base-bne",
    "IsGarrido/roberta-base-bne"
  ],
  "modelo_respaldo": "bertin-project/bertin-roberta-base-spanish",
  "max_seq_length": 128,
  "epocas_techo": 15,
  "seed": 16,
  "n_repeticiones_convergencia": 3
}


## §4 · Reproducibilidad

In [4]:
def fijar_semillas(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)


fijar_semillas(CONFIG["seed"])
print(f"Semilla fijada en {CONFIG['seed']} (random, numpy, tensorflow) -- la misma que la búsqueda bayesiana.")


Semilla fijada en 16 (random, numpy, tensorflow) -- la misma que la búsqueda bayesiana.


## §5 · Carga y limpieza

In [5]:
def cargar_y_limpiar(ruta, cfg):
    col_l, col_t = cfg["col_label"], cfg["col_titulo"]
    col_d, col_f = cfg["col_descripcion"], cfg["col_fecha"]

    df = pd.read_excel(ruta)
    print(f"Filas leídas: {len(df):,}")

    faltantes = {col_l, col_t, col_d} - set(df.columns)
    if faltantes:
        raise ValueError(f"Faltan columnas: {faltantes}. Columnas presentes: {df.columns.tolist()}")

    df = df.copy()
    for c in [col_t, col_d]:
        df[c] = df[c].fillna("").astype(str).str.strip()

    partes = [df[c] for c in cfg["campos_texto"] if c in df.columns]
    df["texto"] = partes[0]
    for p in partes[1:]:
        df["texto"] = (df["texto"] + ". " + p).str.replace(r"\.\s*\.", ".", regex=True)
    df["texto"] = df["texto"].str.strip()

    n0 = len(df)
    df = df[df["texto"].str.len() > 0]

    df[col_l] = pd.to_numeric(df[col_l], errors="coerce")
    if df[col_l].isna().all():
        mapa = {"TRUE": 1, "FALSE": 0, "REAL": 1, "FALSA": 0}
        df[col_l] = df[cfg["col_label"]].astype(str).str.upper().map(mapa)
    df = df[df[col_l].isin([0, 1])]
    df[col_l] = df[col_l].astype(int)
    print(f"Descartadas por texto vacío o etiqueta inválida: {n0 - len(df):,}")

    if col_f in df.columns:
        df[col_f] = pd.to_datetime(df[col_f], errors="coerce", dayfirst=True)

    n1 = len(df)
    df = df.drop_duplicates(subset=["texto", col_l])
    print(f"Duplicados exactos eliminados: {n1 - len(df):,}")

    df = df.reset_index(drop=True)
    df["row_id"] = np.arange(len(df))
    return df


df = cargar_y_limpiar(CONFIG["ruta_xlsx"], CONFIG)
print(f"\nFilas finales: {len(df):,}")
print(df[CONFIG["col_label"]].value_counts())


Filas leídas: 3,259
Descartadas por texto vacío o etiqueta inválida: 0
Duplicados exactos eliminados: 0

Filas finales: 3,259
label
1    1633
0    1626
Name: count, dtype: int64


## §6 · Agrupamiento anti-fuga

In [6]:
def normalizar_texto(s: str) -> str:
    s = unicodedata.normalize("NFKD", str(s)).encode("ascii", "ignore").decode()
    s = s.lower()
    s = re.sub(r"[^a-z0-9 ]", " ", s)
    return re.sub(r"\s+", " ", s).strip()


def asignar_grupos(df, cfg):
    if not cfg["agrupar_variantes"]:
        return np.arange(len(df))

    parent = {}

    def find(x):
        parent.setdefault(x, x)
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x

    def union(a, b):
        ra, rb = find(a), find(b)
        if ra != rb:
            parent[ra] = rb

    tits = df[cfg["col_titulo"]].map(normalizar_texto).values
    descs = df[cfg["col_descripcion"]].map(normalizar_texto).values

    for i in range(len(df)):
        if tits[i]:
            union(("row", i), ("tit", tits[i]))
        if descs[i]:
            union(("row", i), ("des", descs[i]))

    raices = [find(("row", i)) for i in range(len(df))]
    codigo = {r: k for k, r in enumerate(dict.fromkeys(raices))}
    return np.array([codigo[r] for r in raices])


df["grupo"] = asignar_grupos(df, CONFIG)
print(f"Grupos únicos: {df['grupo'].nunique():,} sobre {len(df):,} filas")


Grupos únicos: 3,238 sobre 3,259 filas


## §7 · Partición train / validación / prueba

In [7]:
def _corte_estratificado_por_grupo(y, grupos, fraccion, seed):
    n_splits = max(2, int(round(1 / fraccion)))
    sgkf = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    idx_resto, idx_sel = next(sgkf.split(np.zeros(len(y)), y, groups=grupos))
    return idx_resto, idx_sel


def particionar(df, cfg):
    p_train, p_val, p_test = cfg["proporciones"]
    y = df[cfg["col_label"]].values
    g = df["grupo"].values
    splits = pd.Series("train", index=df.index)

    idx_resto, idx_test = _corte_estratificado_por_grupo(y, g, p_test, cfg["seed"])
    frac_val = p_val / (p_train + p_val)
    idx_train_rel, idx_val_rel = _corte_estratificado_por_grupo(
        y[idx_resto], g[idx_resto], frac_val, cfg["seed"]
    )
    splits.iloc[idx_resto[idx_val_rel]] = "val"
    splits.iloc[idx_test] = "test"
    return splits


df["split"] = particionar(df, CONFIG)
tabla = pd.crosstab(df["split"], df[CONFIG["col_label"]])
tabla["total"] = tabla.sum(axis=1)
print(tabla.loc[["train", "val", "test"]])

n_fuga = int((df.groupby("grupo")["split"].nunique() > 1).sum())
print(f"\nGrupos repartidos entre splits (fuga): {n_fuga}")
print("  ✓ Sin fuga." if n_fuga == 0 else f"  ⚠️ {n_fuga} grupos partidos.")


label     0     1  total
split                   
train  1187  1171   2358
val     221   231    452
test    218   231    449

Grupos repartidos entre splits (fuga): 0
  ✓ Sin fuga.


## §8 · Resolución del modelo preentrenado

In [8]:
try:
    from google.colab import userdata
    _hf_token = userdata.get("HF_TOKEN")
except Exception:
    _hf_token = None

if _hf_token:
    from huggingface_hub import login
    login(token=_hf_token, add_to_git_credential=False)
    print("Autenticado en Hugging Face Hub con el secreto HF_TOKEN de Colab.")
else:
    print("Sin HF_TOKEN en los secretos de Colab: se usan solo repositorios públicos.")

ARCHIVOS_PESOS = ("model.safetensors", "pytorch_model.bin", "tf_model.h5",
                  "model.safetensors.index.json", "pytorch_model.bin.index.json")


def repo_utilizable(repo_id):
    try:
        archivos = set(list_repo_files(repo_id))
    except Exception as e:
        return False, type(e).__name__, None
    if "config.json" not in archivos:
        return False, "sin config.json (deprecado)", None
    if not any(a in archivos for a in ARCHIVOS_PESOS):
        return False, "sin archivos de pesos", None
    try:
        sha = model_info(repo_id).sha
    except Exception:
        sha = None
    return True, "ok", sha


def resolver_modelo(cfg):
    candidatos = list(cfg["modelo_candidatos"]) + [cfg["modelo_respaldo"]]
    print("Comprobando repositorios candidatos:\n")
    for repo in candidatos:
        ok, motivo, sha = repo_utilizable(repo)
        print(f"  {'✓ USABLE' if ok else '✗ ' + motivo:<32} {repo}")
        if ok:
            return repo, sha
    raise RuntimeError("Ningún repositorio candidato tiene pesos descargables.")


MODELO, MODELO_SHA = resolver_modelo(CONFIG)
print(f"\n→ Modelo resuelto: {MODELO}  (commit {MODELO_SHA})")


Sin HF_TOKEN en los secretos de Colab: se usan solo repositorios públicos.
Comprobando repositorios candidatos:



/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


  ✗ sin config.json (deprecado)    PlanTL-GOB-ES/roberta-base-bne
  ✗ RepositoryNotFoundError        BSC-LT/roberta-base-bne
  ✗ RepositoryNotFoundError        BSC-TeMU/roberta-base-bne
  ✓ USABLE                         IsGarrido/roberta-base-bne

→ Modelo resuelto: IsGarrido/roberta-base-bne  (commit 3d54a71ddc029cc1f8dd604e3b7bc0da0e2f7cd9)


## §9 · Tokenización

In [9]:
tokenizer = AutoTokenizer.from_pretrained(MODELO)
print(f"Tokenizador: {tokenizer.__class__.__name__}  |  vocabulario: {tokenizer.vocab_size:,}")


def tokenizar_split(df, split, cfg):
    sub = df[df["split"] == split]
    enc = tokenizer(
        list(sub["texto"].values),
        truncation=True, padding="max_length",
        max_length=cfg["max_seq_length"], return_tensors="np",
    )
    y = sub[cfg["col_label"]].values.astype(np.int32)
    return {"input_ids": enc["input_ids"], "attention_mask": enc["attention_mask"]}, y


X_train, y_train = tokenizar_split(df, "train", CONFIG)
X_val, y_val = tokenizar_split(df, "val", CONFIG)

print(f"train {len(y_train):,} | val {len(y_val):,}")
print("(el conjunto de prueba no se toca en este notebook -- ya se usó una sola vez en finetuning_bayesiano_colombia.ipynb §17)")


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/957 [00:00<?, ?B/s]

Tokenizador: RobertaTokenizerFast  |  vocabulario: 50,262
train 2,358 | val 452
(el conjunto de prueba no se toca en este notebook -- ya se usó una sola vez en finetuning_bayesiano_colombia.ipynb §17)


## §10 · `HyperModel` (mismo que la búsqueda bayesiana)

Se reutiliza literalmente la misma clase `RobertaBNEHyperModel` del notebook
de la búsqueda, para que el modelo que se construye acá sea idéntico
arquitectónicamente al que ganó la búsqueda. Los valores de sus
hiperparámetros se fijan a `MEJORES_HP` (§2) en vez de dejarlos que Keras
Tuner los explore.

In [10]:
def a_lista(x_dict):
    """{"input_ids":..., "attention_mask":...} -> [input_ids, attention_mask]."""
    return [x_dict["input_ids"], x_dict["attention_mask"]]


class F1MacroCallback(tf.keras.callbacks.Callback):
    def __init__(self, x_val, y_val, batch_size=32):
        super().__init__()
        self.x_val = a_lista(x_val)
        self.y_val = y_val
        self.batch_size = batch_size

    def on_epoch_end(self, epoch, logs=None):
        logs = logs or {}
        logits = self.model.predict(self.x_val, batch_size=self.batch_size, verbose=0)
        y_pred = np.argmax(np.asarray(logits), axis=-1)
        logs["val_f1_macro"] = float(f1_score(self.y_val, y_pred, average="macro", zero_division=0))
        logs["val_mcc"] = float(matthews_corrcoef(self.y_val, y_pred))


class RobertaBNEHyperModel(kt.HyperModel):
    def __init__(self, modelo_id, cfg):
        self.modelo_id = modelo_id
        self.cfg = cfg

    def build(self, hp):
        c = self.cfg
        lr = hp.Float("learning_rate", 5e-6, 5e-5, sampling="log", default=MEJORES_HP["learning_rate"])
        dropout = hp.Float("dropout", 0.05, 0.30, default=MEJORES_HP["dropout"])
        l2 = hp.Float("weight_decay", 1e-4, 1e-2, sampling="log", default=MEJORES_HP["weight_decay"])

        config = AutoConfig.from_pretrained(
            self.modelo_id, num_labels=2,
            hidden_dropout_prob=dropout, attention_probs_dropout_prob=dropout,
            classifier_dropout=dropout,
            id2label={0: "falsa", 1: "real"}, label2id={"falsa": 0, "real": 1},
        )
        try:
            backbone = TFAutoModelForSequenceClassification.from_pretrained(
                self.modelo_id, config=config
            )
        except (OSError, EnvironmentError):
            backbone = TFAutoModelForSequenceClassification.from_pretrained(
                self.modelo_id, config=config, from_pt=True
            )
        backbone.trainable = True

        max_len = c["max_seq_length"]
        entrada_ids = tf.keras.Input(shape=(max_len,), dtype=tf.int32, name="input_ids")
        entrada_mask = tf.keras.Input(shape=(max_len,), dtype=tf.int32, name="attention_mask")
        salida = backbone(input_ids=entrada_ids, attention_mask=entrada_mask)
        logits = salida.logits

        modelo = tf.keras.Model(inputs=[entrada_ids, entrada_mask], outputs=logits)

        optimizador = tf.keras.optimizers.AdamW(learning_rate=lr, weight_decay=l2)
        modelo.compile(
            optimizer=optimizador,
            loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
            metrics=["accuracy"],
        )
        return modelo

    def fit(self, hp, model, x, y, validation_data, **kwargs):
        batch_size = hp.Choice("batch_size", [8, 16, 32], default=MEJORES_HP["batch_size"])
        factor_reduccion = hp.Float("lr_reduce_factor", 0.1, 0.5, default=MEJORES_HP["lr_reduce_factor"])
        paciencia_reduccion = hp.Int("lr_reduce_patience", 1, 3, default=MEJORES_HP["lr_reduce_patience"])
        paciencia_parada = hp.Int("early_stopping_patience", 2, 6, default=MEJORES_HP["early_stopping_patience"])

        x_val, y_val = validation_data
        callbacks = [
            F1MacroCallback(x_val, y_val, batch_size=max(32, batch_size * 2)),
            tf.keras.callbacks.ReduceLROnPlateau(
                monitor="val_f1_macro", mode="max",
                factor=factor_reduccion, patience=paciencia_reduccion, min_lr=1e-7,
            ),
            tf.keras.callbacks.EarlyStopping(
                monitor="val_f1_macro", mode="max",
                patience=paciencia_parada, restore_best_weights=True,
            ),
        ] + kwargs.pop("callbacks", [])

        return model.fit(
            a_lista(x), y, batch_size=batch_size, callbacks=callbacks, **kwargs
        )


def construir_hp_fijos(valores):
    """kt.HyperParameters() con .values preseteado -- hp.Float/.Choice/.Int
    de build()/fit() no sobrescriben un valor ya registrado (overwrite=False
    por defecto), asi que esto fija los hiperparametros sin correr una
    busqueda."""
    hp = kt.HyperParameters()
    hp.values = dict(valores)
    return hp


hypermodel = RobertaBNEHyperModel(MODELO, CONFIG)
print("HyperModel listo, con hiperparámetros fijados a los de la búsqueda bayesiana.")


HyperModel listo, con hiperparámetros fijados a los de la búsqueda bayesiana.


## §11 · Callbacks de análisis: pérdida de validación y norma del gradiente

`hypermodel.fit()` no le pasa `validation_data` a `model.fit()` (solo la usa
`F1MacroCallback` para calcular `val_f1_macro`), así que Keras nunca calcula
`val_loss` por su cuenta. `HistorialPerdidaCallback` la calcula a mano,
llamando a `model.evaluate()` sobre el conjunto de validación al final de
cada época.

`NormaGradienteCallback` diagnostica el comportamiento del gradiente durante
el entrenamiento: al final de cada época, calcula el gradiente de la pérdida
sobre un lote fijo de validación y registra su norma L2 global. Una norma que
cae rápido a casi cero sugiere gradientes que se desvanecen; una que crece
sin control sugiere gradientes que explotan.

In [11]:
class HistorialPerdidaCallback(tf.keras.callbacks.Callback):
    def __init__(self, x_val, y_val, batch_size=32):
        super().__init__()
        self.x_val = a_lista(x_val)
        self.y_val = y_val
        self.batch_size = batch_size
        self.historial = {"epoch": [], "loss": [], "val_loss": []}

    def on_epoch_end(self, epoch, logs=None):
        logs = logs or {}
        val_loss = self.model.evaluate(
            self.x_val, self.y_val, batch_size=self.batch_size,
            verbose=0, return_dict=True,
        )["loss"]
        logs["val_loss"] = float(val_loss)
        self.historial["epoch"].append(epoch)
        self.historial["loss"].append(float(logs.get("loss", float("nan"))))
        self.historial["val_loss"].append(float(val_loss))


class NormaGradienteCallback(tf.keras.callbacks.Callback):
    """Calcula la norma del gradiente sobre una muestra chica de
    validacion, en sub-lotes de a lo sumo `chunk_size` ejemplos por
    pasada. GradientTape en modo eager (sin @tf.function) usa bastante
    mas memoria que el train_step compilado de model.fit(), asi que un
    lote tan grande como el de entrenamiento aca puede agotar la GPU --
    de ahi el chunking en vez de una sola pasada con toda la muestra."""
    def __init__(self, x_val, y_val, muestra=16, chunk_size=8, seed=0):
        super().__init__()
        idx = np.random.RandomState(seed).choice(len(y_val), size=min(muestra, len(y_val)), replace=False)
        x_val_lista = a_lista(x_val)
        self.x_muestra = [v[idx] for v in x_val_lista]
        self.y_muestra = y_val[idx]
        self.chunk_size = chunk_size
        self.historial = {"epoch": [], "norma_gradiente_global": []}

    def on_epoch_end(self, epoch, logs=None):
        n = len(self.y_muestra)
        suma_gradientes = None
        n_chunks = 0
        for inicio in range(0, n, self.chunk_size):
            fin = inicio + self.chunk_size
            x_chunk = [v[inicio:fin] for v in self.x_muestra]
            y_chunk = self.y_muestra[inicio:fin]
            with tf.GradientTape() as tape:
                logits = self.model(x_chunk, training=True)
                perdida = self.model.compiled_loss(y_chunk, logits)
            gradientes = tape.gradient(perdida, self.model.trainable_variables)
            if suma_gradientes is None:
                suma_gradientes = [tf.zeros_like(v) for v in self.model.trainable_variables]
            suma_gradientes = [
                s + (g if g is not None else tf.zeros_like(v))
                for s, g, v in zip(suma_gradientes, gradientes, self.model.trainable_variables)
            ]
            n_chunks += 1
            del tape, gradientes, logits, perdida
        gradientes_promedio = [s / n_chunks for s in suma_gradientes]
        norma_global = tf.linalg.global_norm(gradientes_promedio)
        self.historial["epoch"].append(epoch)
        self.historial["norma_gradiente_global"].append(float(norma_global))


print("Callbacks de análisis listos.")


Callbacks de análisis listos.


## §12 · Entrenamiento del modelo final (semilla única)

Entrena **un solo modelo**, con los hiperparámetros de la búsqueda y con
`CONFIG["seed"]` -- la misma semilla que usó la búsqueda bayesiana. Esta es
la sección que produce los gráficos de pérdida y de gradiente.

In [12]:
fijar_semillas(CONFIG["seed"])

hp_final = construir_hp_fijos(MEJORES_HP)
modelo_final = hypermodel.build(hp_final)

_historial_perdida = HistorialPerdidaCallback(X_val, y_val, batch_size=MEJORES_HP["batch_size"])
_norma_gradiente = NormaGradienteCallback(X_val, y_val, muestra=16, chunk_size=min(8, MEJORES_HP["batch_size"]), seed=CONFIG["seed"])

_t0 = time.time()
historial_entrenamiento = hypermodel.fit(
    hp_final, modelo_final,
    X_train, y_train, validation_data=(X_val, y_val),
    epochs=CONFIG["epocas_techo"], verbose=1,
    callbacks=[_historial_perdida, _norma_gradiente],
)
_minutos = (time.time() - _t0) / 60
print(f"\n✓ Entrenamiento del modelo final completo en {_minutos:.1f} minutos.")


config.json:   0%|          | 0.00/613 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/499M [00:00<?, ?B/s]

TensorFlow and JAX classes are deprecated and will be removed in Transformers v5. We recommend migrating to PyTorch classes or pinning your version of Transformers.
Some weights of the PyTorch model were not used when initializing the TF 2.0 model TFRobertaForSequenceClassification: ['roberta.embeddings.position_ids']
- This IS expected if you are initializing TFRobertaForSequenceClassification from a PyTorch model trained on another task or with another architecture (e.g. initializing a TFBertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing TFRobertaForSequenceClassification from a PyTorch model that you expect to be exactly identical (e.g. initializing a TFBertForSequenceClassification model from a BertForSequenceClassification model).
Some weights or buffers of the TF 2.0 model TFRobertaForSequenceClassification were not initialized from the PyTorch model and are newly initialized: ['classifier.dense.weight', 'classifie

Epoch 1/15
295/295 [==============================] - ETA: 0s - loss: 0.6748 - accuracy: 0.5606

ResourceExhaustedError: Exception encountered when calling layer 'intermediate' (type TFRobertaIntermediate).

{{function_node __wrapped__Neg_device_/job:localhost/replica:0/task:0/device:GPU:0}} failed to allocate memory [Op:Neg] name: 

Call arguments received by layer 'intermediate' (type TFRobertaIntermediate):
  • hidden_states=tf.Tensor(shape=(64, 128, 768), dtype=float32)

## §13 · Gráficos: pérdida por época y comportamiento del gradiente

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
epocas_x = _historial_perdida.historial["epoch"]
ax.plot(epocas_x, _historial_perdida.historial["loss"], marker="o", label="Pérdida de entrenamiento")
ax.plot(epocas_x, _historial_perdida.historial["val_loss"], marker="o", label="Pérdida de validación")
ax.set_xlabel("Época")
ax.set_ylabel("Pérdida (entropía cruzada)")
ax.set_title("Modelo final -- pérdida de entrenamiento vs. validación por época")
ax.legend()
ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig(os.path.join(DIR_SALIDA, "perdida_train_val.png"), dpi=150)
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(
    _norma_gradiente.historial["epoch"],
    _norma_gradiente.historial["norma_gradiente_global"],
    marker="o", color="darkred",
)
ax.set_xlabel("Época")
ax.set_ylabel("Norma L2 global del gradiente")
ax.set_title("Modelo final -- comportamiento del gradiente por época")
ax.set_yscale("log")
ax.grid(alpha=0.3, which="both")
fig.tight_layout()
fig.savefig(os.path.join(DIR_SALIDA, "norma_gradiente.png"), dpi=150)
plt.show()

print("Norma del gradiente por época:")
for e, n in zip(_norma_gradiente.historial["epoch"], _norma_gradiente.historial["norma_gradiente_global"]):
    print(f"  época {e:>2}: {n:.4f}")


## §14 · Prueba de convergencia de los hiperparámetros (sección independiente)

**Esta sección no tiene relación con el modelo final entrenado arriba.**
Reentrena los mismos hiperparámetros de la búsqueda `n_repeticiones_convergencia`
veces, cada vez con una semilla distinta, para comprobar si el resultado de
la búsqueda bayesiana es reproducible o fue un intento con suerte. Es la
**única** sección de este notebook que usa más de una semilla; todo lo demás
usa exclusivamente `CONFIG["seed"]`.

Se evalúa sobre el conjunto de **validación**, nunca sobre prueba.

In [ ]:
def calcular_metricas(y_true, y_pred):
    m = {
        "accuracy": accuracy_score(y_true, y_pred),
        "f1_macro": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "precision_macro": precision_score(y_true, y_pred, average="macro", zero_division=0),
        "recall_macro": recall_score(y_true, y_pred, average="macro", zero_division=0),
        "mcc": matthews_corrcoef(y_true, y_pred),
    }
    return {k: float(round(v, 4)) for k, v in m.items()}


SEMILLAS_CONVERGENCIA = [CONFIG["seed"] + i for i in range(CONFIG["n_repeticiones_convergencia"])]
print(f"Semillas de la prueba de convergencia (independientes del resto del notebook): {SEMILLAS_CONVERGENCIA}")

resultados_convergencia = []
for semilla in SEMILLAS_CONVERGENCIA:
    print(f"\n{'='*60}\nRepetición con semilla {semilla}\n{'='*60}")
    fijar_semillas(semilla)

    hp_rep = construir_hp_fijos(MEJORES_HP)
    modelo_rep = hypermodel.build(hp_rep)
    hypermodel.fit(
        hp_rep, modelo_rep,
        X_train, y_train, validation_data=(X_val, y_val),
        epochs=CONFIG["epocas_techo"], verbose=0,
    )

    logits_val = modelo_rep.predict(a_lista(X_val), batch_size=32, verbose=0)
    y_pred_val = np.argmax(np.asarray(logits_val), axis=-1)
    metricas = calcular_metricas(y_val, y_pred_val)
    metricas["semilla"] = semilla
    resultados_convergencia.append(metricas)
    print(f"  val_f1_macro = {metricas['f1_macro']:.4f}")

    del modelo_rep
    tf.keras.backend.clear_session()

df_convergencia = pd.DataFrame(resultados_convergencia).set_index("semilla")
print("\nResultados por semilla:\n")
print(df_convergencia)


In [ ]:
media_f1 = df_convergencia["f1_macro"].mean()
desvio_f1 = df_convergencia["f1_macro"].std()

print(f"val_f1_macro -- media: {media_f1:.4f}  |  desvío estándar: {desvio_f1:.4f}")

UMBRAL_DESVIO_BAJO = 0.02
if desvio_f1 < UMBRAL_DESVIO_BAJO:
    print(f"\n✓ Desvío bajo (< {UMBRAL_DESVIO_BAJO}): el resultado de la búsqueda bayesiana parece "
          "reproducible, no un intento con suerte.")
else:
    print(f"\n⚠️ Desvío alto (>= {UMBRAL_DESVIO_BAJO}): el resultado varía bastante entre semillas -- "
          "tomar la mejor combinación encontrada con cautela.")

fig, ax = plt.subplots(figsize=(7, 5))
ax.bar([str(s) for s in df_convergencia.index], df_convergencia["f1_macro"], color="steelblue")
ax.axhline(media_f1, color="black", linestyle="--", label=f"Media = {media_f1:.4f}")
ax.set_xlabel("Semilla")
ax.set_ylabel("val_f1_macro")
ax.set_title("Prueba de convergencia -- val_f1_macro por semilla")
ax.legend()
ax.grid(alpha=0.3, axis="y")
fig.tight_layout()
fig.savefig(os.path.join(DIR_SALIDA, "convergencia_f1_por_semilla.png"), dpi=150)
plt.show()


## §15 · Guardado de resultados

In [ ]:
df_convergencia.reset_index().to_csv(
    os.path.join(DIR_SALIDA, "convergencia_por_semilla.csv"), index=False
)

pd.DataFrame(_historial_perdida.historial).to_csv(
    os.path.join(DIR_SALIDA, "perdida_modelo_final.csv"), index=False
)
pd.DataFrame(_norma_gradiente.historial).to_csv(
    os.path.join(DIR_SALIDA, "norma_gradiente_modelo_final.csv"), index=False
)

with open(os.path.join(DIR_SALIDA, "resumen_analisis.json"), "w", encoding="utf-8") as f:
    json.dump({
        "semilla_modelo_final": CONFIG["seed"],
        "semillas_convergencia": SEMILLAS_CONVERGENCIA,
        "hiperparametros": MEJORES_HP,
        "convergencia_val_f1_macro_media": float(media_f1),
        "convergencia_val_f1_macro_desvio": float(desvio_f1),
        "fecha": datetime.now().isoformat(),
    }, f, indent=2, ensure_ascii=False)

print(f"Guardado en ./{DIR_SALIDA}/:")
print("  - perdida_train_val.png / perdida_modelo_final.csv")
print("  - norma_gradiente.png / norma_gradiente_modelo_final.csv")
print("  - convergencia_f1_por_semilla.png / convergencia_por_semilla.csv")
print("  - resumen_analisis.json")
